In [4]:
import pandas as pd
import numpy as np
import re
import xarray as xr
import warnings

warnings.filterwarnings('ignore')

In [5]:
df = pd.read_excel(r'Dane\PositionReport.xlsx', header=[8,9])

In [6]:
czyste_kolumny = []

for col in df.columns:
    czesc_gorna = str(col[0]) if 'Unnamed' not in str(col[0]) else ''
    czesc_dolna = str(col[1]) if 'Unnamed' not in str(col[1]) else ''
    pelna_nazwa = f"{czesc_gorna} {czesc_dolna}".replace('\n', ' ').strip()
    pelna_nazwa = pelna_nazwa.replace('  ', ' ')
    czyste_kolumny.append(pelna_nazwa)
df.columns = czyste_kolumny



In [7]:

def dms_to_dd(dms_str):
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
        
    return round(dd, 5)
df.columns = df.columns.str.strip()
df['Time'] = pd.to_datetime(df['Time'], format='%d.%m.%Y %H:%M')
df['Lat_dd'] = df['Lat'].apply(dms_to_dd)
df['Lon_dd'] = df['Lon'].apply(dms_to_dd)

df_clean = df.dropna(subset=['Speed [kn]']).copy()
df_clean = df_clean[df_clean['Speed [kn]'] > 0]
print(f"Liczba wierszy przed czyszczeniem: {len(df)}")
print(f"Liczba wierszy po odcięciu postojów: {len(df_clean)}")
display(df_clean[['Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]']].head())

Liczba wierszy przed czyszczeniem: 156942
Liczba wierszy po odcięciu postojów: 132042


,Time,Lat_dd,Lon_dd,Speed [kn],Course [°]
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0


In [8]:
print("Bounding Box:")
print(f"Północ (North): {df_clean['Lat_dd'].max()}")
print(f"Południe (South): {df_clean['Lat_dd'].min()}")
print(f"Wschód (East): {df_clean['Lon_dd'].max()}")
print(f"Zachód (West): {df_clean['Lon_dd'].min()}")
print(f"Początek (Od): {df_clean['Time'].min()}")
print(f"Koniec (Do): {df_clean['Time'].max()}")

Bounding Box:
Północ (North): 61.23332
Południe (South): 51.03923
Wschód (East): 11.20088
Zachód (West): -2.12149
Początek (Od): 2023-02-02 00:06:00
Koniec (Do): 2026-02-02 08:34:57


In [ ]:
df.head()

,Time,Lat,Lon,Speed [kn],Calculated Speed [kn],Course [°],Distance since last point [nm],Total distance run [nm],Lat_dd,Lon_dd
0,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000,56.82667,-0.12833
1,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126,56.88667,-0.12500
2,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728,56.92333,-0.05833
3,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653,56.98667,0.05500
4,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580,57.05000,0.17500
